# Speech-to-Speech: Arabisch → Deutsch (Audio rein, Audio raus)

**Pipeline:** Arabische Sprache → **Cohere Transcribe Arabic** → **TranslateGemma 12B** → **F5-TTS German** → deutsches Audio.

Kein Text eintippen nötig: Du sprichst Arabisch ins Mikrofon (oder lädst eine Datei hoch) und bekommst direkt deutsches Audio zurück.

**Start:** `Laufzeit > Laufzeittyp ändern > T4 GPU`, dann `Laufzeit > Alle ausführen`.

1. `HF_TOKEN` als Colab-Secret anlegen (Schlüssel-Symbol links) oder `.env` hochladen, wenn gefragt. Die Bedingungen von `CohereLabs/cohere-transcribe-arabic-07-2026` und `google/translategemma-12b-it` müssen auf huggingface.co akzeptiert sein.
2. Zelle 3 lädt alle drei Modelle **einmal** in die GPU.
3. Zelle 4 nimmt auf (Mikrofon-Erlaubnis im Browser bestätigen), übersetzt und spielt das deutsche Audio automatisch ab.
4. **Für jeden neuen Satz nur Zelle 4 erneut ausführen** (Play-Symbol) – die Modelle bleiben geladen.

In [ ]:
#@title 1. GPU prüfen und Pakete installieren
import subprocess, sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Keine GPU: Laufzeit > Laufzeittyp ändern > 'T4 GPU' > Speichern, dann 'Laufzeit > Alle ausführen'. "
        "Bleibt der Fehler, ist das kostenlose GPU-Kontingent für heute aufgebraucht."
    )
print("GPU:", torch.cuda.get_device_name(0))

PACKAGES = [
    "transformers>=5.4.0", "accelerate", "bitsandbytes>=0.45", "f5-tts", "huggingface_hub",
    "soundfile", "librosa", "sentencepiece", "protobuf", "python-dotenv",
]
IMPORT_CHECK = (
    "import f5_tts.api, soundfile, librosa, bitsandbytes, dotenv; "
    "from transformers import AutoProcessor, CohereAsrForConditionalGeneration, AutoModelForImageTextToText"
)

result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input", *PACKAGES],
                        capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-3000:])
    print(result.stderr[-3000:])
    raise RuntimeError("Installation fehlgeschlagen (siehe oben).")

check = subprocess.run([sys.executable, "-c", IMPORT_CHECK], capture_output=True, text=True)
if check.returncode != 0:
    print(check.stderr[-3000:])
    raise RuntimeError("Pakete installiert, aber nicht importierbar (siehe oben).")
print("OK: Pakete installiert.")

In [ ]:
#@title 2. Hugging Face Token
import os
from pathlib import Path

from dotenv import dotenv_values
from huggingface_hub import hf_hub_download, login

ASR_MODEL = "CohereLabs/cohere-transcribe-arabic-07-2026"
MT_MODEL = "google/translategemma-12b-it"


def find_token():
    try:
        from google.colab import userdata
        token = (userdata.get("HF_TOKEN") or "").strip()
        if token:
            return token
    except Exception:
        pass
    for path in [Path("/content/.env"), *sorted(Path("/content").glob("*env*"))]:
        if path.is_file():
            token = (dotenv_values(path).get("HF_TOKEN") or "").strip()
            if token:
                return token
    from google.colab import files
    print("Kein HF_TOKEN gefunden. Bitte die .env mit HF_TOKEN=... hochladen:")
    for name in files.upload():
        token = (dotenv_values(Path("/content") / Path(name).name).get("HF_TOKEN") or "").strip()
        if token:
            return token
    return ""


HF_TOKEN = find_token()
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN fehlt: als Colab-Secret anlegen oder eine .env mit HF_TOKEN=... hochladen.")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

for repo in (ASR_MODEL, MT_MODEL):
    try:
        hf_hub_download(repo, "config.json", token=HF_TOKEN)
    except Exception as exc:
        raise RuntimeError(
            f"Kein Zugriff auf {repo}: auf huggingface.co/{repo} die Bedingungen akzeptieren "
            f"und prüfen, ob der Token Leserechte hat. ({exc})"
        ) from exc
print("OK: Token gültig.")

In [ ]:
#@title 3. Alle drei Modelle laden (einmalig)
import re
from importlib.resources import files as package_files

import numpy as np
import torch
from f5_tts.api import F5TTS
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
    CohereAsrForConditionalGeneration,
)

SAMPLE_RATE = 16_000
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
F5_GERMAN_REPO = "aihpi/F5-TTS-German"
F5_GERMAN_CKPT = "F5TTS_Base/model_420000.safetensors"
# Built-in F5-TTS voice, so no reference text has to be typed.
REF_WAV = str(package_files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav"))
REF_TEXT = "Some call me nature, others call me mother nature."

print("1/3 Lade Cohere Transcribe Arabic ...")
asr_processor = AutoProcessor.from_pretrained(ASR_MODEL, token=HF_TOKEN)
asr_model = CohereAsrForConditionalGeneration.from_pretrained(
    ASR_MODEL, dtype=DTYPE, device_map="auto", token=HF_TOKEN
).eval()

print("2/3 Lade TranslateGemma 12B (4-bit) ...")
mt_processor = AutoProcessor.from_pretrained(MT_MODEL, token=HF_TOKEN)
mt_model = AutoModelForImageTextToText.from_pretrained(
    MT_MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE,
    ),
    device_map="auto",
    token=HF_TOKEN,
).eval()

print("3/3 Lade F5-TTS German ...")
tts = F5TTS(
    model="F5TTS_Base",
    ckpt_file=hf_hub_download(F5_GERMAN_REPO, F5_GERMAN_CKPT),
    vocab_file=hf_hub_download(F5_GERMAN_REPO, "vocab.txt"),
    device="cuda:0",
)


def transcribe(audio):
    inputs = asr_processor(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt", language="ar")
    chunk_index = inputs.get("audio_chunk_index")
    inputs = inputs.to(asr_model.device, dtype=asr_model.dtype)
    with torch.inference_mode():
        outputs = asr_model.generate(**inputs, max_new_tokens=256)
    decoded = asr_processor.decode(outputs, skip_special_tokens=True, audio_chunk_index=chunk_index, language="ar")
    return (" ".join(decoded) if isinstance(decoded, list) else decoded).strip()


def split_sentences(text, max_chars=600):
    sentences = [s.strip() for s in re.split(r"(?<=[.!?؟])\s+|\n+", text) if s.strip()]
    chunks, current = [], ""
    for sentence in sentences:
        if current and len(current) + len(sentence) + 1 > max_chars:
            chunks.append(current)
            current = sentence
        else:
            current = f"{current} {sentence}".strip()
    if current:
        chunks.append(current)
    return chunks


def translate(text):
    results = []
    for chunk in split_sentences(text):
        messages = [{
            "role": "user",
            "content": [{"type": "text", "source_lang_code": "ar", "target_lang_code": "de", "text": chunk}],
        }]
        batch = mt_processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
        ).to(mt_model.device)
        input_len = batch["input_ids"].shape[1]
        with torch.inference_mode():
            output = mt_model.generate(**batch, max_new_tokens=512, do_sample=False)
        results.append(mt_processor.decode(output[0, input_len:], skip_special_tokens=True).strip())
    return " ".join(results).strip()


def speak(text):
    wav, rate, _ = tts.infer(ref_file=REF_WAV, ref_text=REF_TEXT, gen_text=text, nfe_step=32, remove_silence=False)
    return np.asarray(wav, dtype=np.float32).reshape(-1), rate


print(f"OK: alle Modelle geladen ({torch.cuda.memory_allocated() / 1e9:.1f} GB GPU belegt).")

In [ ]:
#@title 4. Arabisch sprechen → deutsches Audio (für jeden neuen Satz erneut ausführen)
QUELLE = "Mikrofon"  #@param ["Mikrofon", "Datei hochladen"]
AUFNAHME_SEKUNDEN = 10  #@param {type:"slider", min:3, max:60, step:1}

import subprocess
import time
from base64 import b64decode

import librosa
import soundfile as sf
from google.colab import files, output
from IPython.display import Audio, Javascript, display

RECORD_JS = """
async function recordAudio(ms) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(resolve => recorder.onstop = resolve);
  recorder.start();
  await new Promise(resolve => setTimeout(resolve, ms));
  recorder.stop();
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  const blob = new Blob(chunks);
  return await new Promise(resolve => {
    const reader = new FileReader();
    reader.onloadend = () => resolve(reader.result);
    reader.readAsDataURL(blob);
  });
}
"""

if QUELLE == "Mikrofon":
    display(Javascript(RECORD_JS))
    print(f"Aufnahme läuft {AUFNAHME_SEKUNDEN} s – jetzt Arabisch sprechen ...")
    data_url = output.eval_js(f"recordAudio({AUFNAHME_SEKUNDEN * 1000})")
    Path("/content/aufnahme.webm").write_bytes(b64decode(data_url.split(",", 1)[1]))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/content/aufnahme.webm",
                    "-ac", "1", "-ar", str(SAMPLE_RATE), "/content/aufnahme.wav"], check=True)
    input_path = "/content/aufnahme.wav"
else:
    print("Arabische Audiodatei auswählen (wav, mp3, m4a, ...):")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Keine Datei hochgeladen.")
    input_path = str(Path("/content") / Path(next(iter(uploaded))).name)

audio_ar, _ = librosa.load(input_path, sr=SAMPLE_RATE, mono=True)
print(f"Eingabe: {len(audio_ar) / SAMPLE_RATE:.1f} s")

start = time.time()
text_ar = transcribe(audio_ar)
if not text_ar:
    raise RuntimeError("Keine arabische Sprache erkannt – bitte lauter/näher am Mikrofon sprechen.")
t_asr = time.time()
text_de = translate(text_ar)
t_mt = time.time()
wav_de, tts_rate = speak(text_de)
t_tts = time.time()

output_wav = f"/content/deutsch_{time.strftime('%H%M%S')}.wav"
sf.write(output_wav, wav_de, tts_rate)

print(f"\nArabisch: {text_ar}")
print(f"Deutsch:  {text_de}")
print(f"\nZeit: ASR {t_asr - start:.1f} s | Übersetzung {t_mt - t_asr:.1f} s | TTS {t_tts - t_mt:.1f} s")
print("Gespeichert:", output_wav)
display(Audio(wav_de, rate=tts_rate, autoplay=True))